# 08 - Leakage-safe publication data

CPU stage. Range-sample the canonical Drive CSV sources, remove duplicate and identifier leakage, split first, and fit preprocessing on training rows only.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').is_file() and (path / 'src' / 'rareguard').is_dir()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate

    if Path('/content').is_dir():
        from google.colab import drive

        drive.mount('/content/drive', force_remount=False)
        for candidate in DRIVE_CANDIDATES:
            if is_project_root(candidate):
                return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'RareGuard-XIDS IMPLEMENTATION project not found. Expected the canonical '
        'Google Drive project at '
        '/content/drive/MyDrive/projects/implementation/'
        'RareGuard-XIDS[DeepakSir]/IMPLEMENTATION or the canonical local project.'
    )
os.chdir(PROJECT_ROOT)
PROJECT_SRC = str(PROJECT_ROOT / 'src')
sys.path.insert(0, PROJECT_SRC)
os.environ['PYTHONPATH'] = PROJECT_SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Project root:', PROJECT_ROOT)
print('Runtime:', 'Colab' if Path('/content').is_dir() else 'local')
print('Python path ready')


Project root: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Runtime: Colab
Python path ready


In [2]:
!python -m rareguard.experiments.prepare_publication_data --config configs/publication.yaml --datasets configs/datasets.yaml --dataset all --resume


2026-10-23 14:04:04,029 | INFO | rareguard | Skipping completed publication split cse_cic_ids2018
2026-10-23 14:04:05,005 | INFO | rareguard | Skipping completed publication split cic_bccc_nrc_tabulariotattack_2024
2026-10-23 14:04:05,513 | INFO | rareguard | Skipping completed publication split cic_iiot_2025_datasense
2026-10-23 14:04:05,946 | INFO | rareguard | Skipping completed publication split edge_iiotset


In [3]:
from pathlib import Path
import json
for path in sorted(Path('artifacts/publication/processed').glob('*/split_metadata.json')):
    data=json.loads(path.read_text())
    print(path.parent.name, data['split_rows'], 'features=', data['features'], 'classes=', data['classes'], 'ready=', data['ready'])


cic_bccc_nrc_tabulariotattack_2024 {'train': 24829, 'validation': 3548, 'test': 7095} features= 79 classes= 15 ready= True
cic_iiot_2025_datasense {'train': 9099, 'validation': 1300, 'test': 2600} features= 71 classes= 39 ready= True
cse_cic_ids2018 {'train': 14794, 'validation': 2114, 'test': 4228} features= 78 classes= 13 ready= True
edge_iiotset {'train': 12187, 'validation': 1742, 'test': 3483} features= 53 classes= 11 ready= True
